### **Step 1 - Get The database.json file**
Go to the github page for CutePetsBoston.  Open Actions.  Grab the artifatcs from the last run.  Download database.json and drop it into the data_sandbox folder.

In [1]:
import io
import json
import zipfile
import pandas as pd
import requests

### **Step 2 - Fake database.py file**
This is a fake *database.py* file.  **Do not run this notebook in the main folder.** The point here is to download *database.json* from the latest artefact in order to have data to play with; this will allow your testing version of *dashboard.py* to pull data properl;y without needing to be changed, only moved to this folder.

In [2]:
%%writefile database.py
import io
import json
import logging
import traceback
import urllib.request
import zipfile
from pathlib import Path
import pandas as pd
import requests


def read_database(database_path):
    path = Path(database_path)
    if not path.exists() or path.stat().st_size == 0:
        return {}
    try:
        with path.open() as database_file:
            return json.load(database_file)
    except (json.JSONDecodeError, ValueError) as exc:
        logger.error("%s:%s", type(exc).__name__, exc)
        traceback.print_exc()
        return {}


Writing database.py


### **Step 3 - Profit?**
Do some data analytics, have a blast.  Figure out what you want to do.

In [5]:
# then we can run this...

from database import read_database
# Load the specific table into Pandas

data = read_database(database_path='database.json')


df_posts = pd.json_normalize(
    data["posts"],
    record_path=["metrics"],  # Unpacks the nested metrics array
    meta=[
        "pet_id",
        "platform",
        "post_id",
        "post_url",
        "posted_at",
    ],  # Carries along parent post info
)

# Create DataFrames for both sections
df_pets = pd.DataFrame(data["posted_pets"])

df_posts = pd.json_normalize(
    data["posts"],
    record_path=["metrics"],
    meta=["pet_id", "platform", "post_id", "post_url"],
)

# Merge pets metadata with detailed post metrics
df_metrics = pd.merge(df_pets, df_posts, on="pet_id", how="left")
df_metrics.head()


,name,pet_id,posted_at,collected_at,likes,reposts,comments,platform,post_id,post_url
0,Kia Stanton,22051034,2026-10-05T12:11:13.242016+00:00,2026-10-05T12:11:13.829996+00:00,0,0.0,2,Mastodon,117388229311534085,https://mastodon.social/@cutepetsboston/117388...
1,Kia Stanton,22051034,2026-10-05T12:11:13.242016+00:00,2026-10-06T00:17:30.388021+00:00,0,0.0,2,Mastodon,117388229311534085,https://mastodon.social/@cutepetsboston/117388...
2,Kia Stanton,22051034,2026-10-05T12:11:13.242016+00:00,2026-10-06T04:08:16.332500+00:00,0,0.0,2,Mastodon,117388229311534085,https://mastodon.social/@cutepetsboston/117388...
3,Kia Stanton,22051034,2026-10-05T12:11:13.242016+00:00,2026-10-06T08:09:36.959428+00:00,0,0.0,2,Mastodon,117388229311534085,https://mastodon.social/@cutepetsboston/117388...
4,Kia Stanton,22051034,2026-10-05T12:11:13.242016+00:00,2026-10-06T12:10:22.901751+00:00,0,0.0,2,Mastodon,117388229311534085,https://mastodon.social/@cutepetsboston/117388...


In [11]:
df_metrics["day"] = pd.to_datetime(df_metrics["posted_at"]).dt.tz_localize(None).dt.normalize()
list(set(df_metrics["day"]))

[Timestamp('2026-10-06 00:00:00'), Timestamp('2026-10-05 00:00:00')]

### **Step 4 - mterics_dashboard.py**
This is where we take the above and incorporate it into *medashboard.py.  In fact, you may want to make sure that dashboard.py is the latest version (copy paste from main folder)

In [8]:
%%writefile metrics_dashboard.py
import pandas as pd
import plotly.express as px
from database import read_database

def dashboard(html_file_path="dashboard.html", database_path="database.json"):
    # 1. Fetch and prepare data
    data = read_database(database_path)
    
    df_pets = pd.DataFrame(data.get("posted_pets", []))
    df_posts = pd.json_normalize(
        data.get("posts", []),
        record_path=["metrics"],
        meta=["pet_id", "platform", "post_id", "post_url"],
    )
    
    df_metrics = pd.merge(df_pets, df_posts, on="pet_id", how="left")
    
    # Process views daily aggregations
    df_metrics["day"] = pd.to_datetime(df_metrics["posted_at"]).dt.tz_localize(None).dt.normalize()
    views = (
        df_metrics[["day", "platform", "post_id", "likes", "reposts", "comments"]]
        .groupby(["day", "platform", "post_id"])
        .max()
        .reset_index()
    )
    views = views.groupby(["day", "platform"])[["likes", "reposts", "comments"]].sum().reset_index()

    # 2. Options Setup
    platforms_html = ["All Platforms"] + sorted(
        [str(p) for p in df_metrics["platform"].dropna().unique() if p != "All Platforms"]
    )
    time_ranges = {
        "7d": "Last 7 Days",
        "28d": "Last 28 Days",
        "1y": "Last Year",
        "all": "All Time",
    }

    # 3. Build Dropdowns
    platform_options_html = "".join(
        [f'<option value="{p}">{p}</option>' for p in platforms_html]
    )
    timerange_options_html = "".join(
        [f'<option value="{key}">{label}</option>' for key, label in time_ranges.items()]
    )

    # 4. Determine max date and cutoff windows
    max_date = views["day"].max() if not views.empty else pd.Timestamp.now().normalize()
    cutoff_dates = {
        "all": None,
        "7d": max_date - pd.Timedelta(days=6),
        "28d": max_date - pd.Timedelta(days=27),
        "1y": max_date - pd.Timedelta(days=364),
    }

    # 5. Build HTML Sections
    sections_html = ""
    for p in platforms_html:
        p_df = df_metrics if p == "All Platforms" else df_metrics[df_metrics["platform"] == p]
        
        # --- SEPARATE TOP 5 CHARTS FOR LIKES, REPOSTS, COMMENTS ---
        top_charts_html = ""
        for metric in ["likes", "reposts", "comments"]:
            top_pets = (
                p_df.groupby(["pet_id", "name"])[metric]
                .sum()
                .reset_index()
                .sort_values(by=metric, ascending=False)
                .head(5)
            )
            
            fig_top = px.bar(
                top_pets,
                x="name",
                y=metric,
                title=f"Top 5 Pets by {metric.capitalize()} ({p})",
                labels={"name": "Pet Name", metric: f"Total {metric.capitalize()}"},
            )
            top_charts_html += fig_top.to_html(full_html=False, include_plotlyjs=False)

        # --- METRICS CHARTS PER TIME RANGE ---
        p_views = views if p == "All Platforms" else views[views["platform"] == p]

        metrics_html_by_range = ""
        for range_key in time_ranges.keys():
            cutoff = cutoff_dates[range_key]
            
            if cutoff is not None and not p_views.empty:
                full_dates = pd.date_range(start=cutoff, end=max_date, freq="D")
                unique_platforms = p_views["platform"].unique()
                idx = pd.MultiIndex.from_product([full_dates, unique_platforms], names=["day", "platform"])
                
                filtered_views = (
                    p_views.set_index(["day", "platform"])
                    .reindex(idx, fill_value=0)
                    .reset_index()
                )
            else:
                filtered_views = p_views

            metrics_charts = ""
            for metric in ["likes", "reposts", "comments"]:
                fig_metric = px.line(
                    filtered_views,
                    x="day",
                    y=metric,
                    color="platform" if p == "All Platforms" else None,
                    title=f"Total {metric.capitalize()} Over Time ({p} - {time_ranges[range_key]})",
                    labels={"day": "Date", metric: metric.capitalize()},
                )
                fig_metric.update_xaxes(tickangle=45, dtick="D1" if range_key == "7d" else None)
                metrics_charts += fig_metric.to_html(full_html=False, include_plotlyjs=False)

            is_default_range = "block" if range_key == "7d" else "none"
            metrics_html_by_range += f"""
            <div id="metrics-{p}-{range_key}" class="metrics-range-section metrics-{p}" style="display: {is_default_range};">
                {metrics_charts}
            </div>
            """

        is_default_platform = "block" if p == "All Platforms" else "none"
        sections_html += f"""
        <div id="platform-{p}" class="platform-section" style="display: {is_default_platform};">
            <h2>Top 5 Overall</h2>
            {top_charts_html}
            <hr style="margin: 30px 0;">
            <h3>Metrics Over Time</h3>
            <div style="margin-bottom: 20px;">
                <label for="timerange-select"><strong>Date Range:</strong> </label>
                <select id="timerange-select" onchange="updateDashboard()">
                    {timerange_options_html}
                </select>
            </div>
            {metrics_html_by_range}
        </div>
        """

    # 6. Construct complete HTML file
    full_html = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>Cute Pets Boston Metrics Dashboard</title>
        <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 20px; }}
            .controls {{ margin-bottom: 25px; }}
            select {{ padding: 8px; font-size: 16px; }}
        </style>
    </head>
    <body>
        <h1>Cute Pets Boston Top Pets & Metrics</h1>
        
        <div class="controls">
            <label for="platform-select"><strong>Platform:</strong> </label>
            <select id="platform-select" onchange="updateDashboard()">
                {platform_options_html}
            </select>
        </div>

        {sections_html}

        <script>
            function updateDashboard() {{
                var selectedPlatform = document.getElementById('platform-select').value;
                var activePlatform = document.getElementById('platform-' + selectedPlatform);
                
                var activeRangeSelect = activePlatform.querySelector('#timerange-select');
                var selectedRange = activeRangeSelect ? activeRangeSelect.value : '7d';

                var platformSections = document.getElementsByClassName('platform-section');
                for (var i = 0; i < platformSections.length; i++) {{
                    platformSections[i].style.display = 'none';
                }}
                if (activePlatform) {{
                    activePlatform.style.display = 'block';
                }}

                var metricSections = document.getElementsByClassName('metrics-range-section');
                for (var j = 0; j < metricSections.length; j++) {{
                    metricSections[j].style.display = 'none';
                }}
                var activeMetrics = document.getElementById('metrics-' + selectedPlatform + '-' + selectedRange);
                if (activeMetrics) {{
                    activeMetrics.style.display = 'block';
                }}
            }}
        </script>
    </body>
    </html>
    """

    with open(html_file_path, "w", encoding="utf-8") as f:
        f.write(full_html)

    print(f"Dashboard saved to {html_file_path}")

Overwriting metrics_dashboard.py


### **Step 6 - Run This to See Your Results**
Do not save the py file until you have it working correctly.

In [10]:
from metrics_dashboard import dashboard 

dashboard(html_file_path="dashboard.html", database_path="database.json")

Dashboard saved to dashboard.html
